In [1]:
import pandas as pd
import numpy as np

In [2]:
df= pd.read_excel('marketing_campaign_dataset-cleaned.xlsx')

In [18]:
df.head(10)

,Campaign_ID,Campaign_Type,Target_Audience,Channel_Used,Conversion %,Acquisition_Cost,ROI %,Location,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Email,Men 18-24,Google Ads,4,16174,6.29,Chicago,506,1922,6,Health & Wellness,2021-01-01
1,2,Email,Women 35-44,Google Ads,12,11566,5.61,New York,116,7523,7,Fashionistas,2021-01-02
2,3,Influencer,Men 25-34,YouTube,7,10200,7.18,Los Angeles,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,Display,All Ages,YouTube,11,12724,5.55,Miami,217,1820,7,Health & Wellness,2021-01-04
4,5,Email,Men 25-34,YouTube,5,16452,6.50,Los Angeles,379,4201,3,Health & Wellness,2021-01-05
5,6,Display,All Ages,Instagram,7,9716,4.36,New York,100,1643,1,Foodies,2021-01-06
6,7,Email,Women 35-44,Website,13,11067,2.86,Los Angeles,817,8749,10,Tech Enthusiasts,2021-01-07
7,8,Search,Men 18-24,Google Ads,8,13280,5.55,Los Angeles,624,7854,7,Outdoor Adventurers,2021-01-08
8,9,Social Media,Women 35-44,Facebook,9,18066,6.73,Chicago,861,1754,6,Tech Enthusiasts,2021-01-09
9,10,Email,Women 35-44,Instagram,9,13766,3.78,Los Angeles,642,3856,3,Tech Enthusiasts,2021-01-10


In [10]:
# 1. Structure: expected columns and data types
expected_cols = ["Campaign_ID", "Campaign_Type", "Target_Audience", "Channel_Used",
                 "Conversion %", "Acquisition_Cost", "ROI %", "Location",
                 "Clicks","Impressions", "Engagement_Score", "Customer_Segment", "Date"]
assert list(df.columns) == expected_cols, "Column names/order differ"
assert pd.api.types.is_datetime64_any_dtype(df["Date"])
assert pd.api.types.is_numeric_dtype(df["Acquisition_Cost"])

# 2. Completeness: no missing values
assert df.isna().sum().sum() == 0

# 3. Uniqueness: no duplicate rows or IDs
assert not df.duplicated().any()
assert df["Campaign_ID"].is_unique

# 4. Range checks (numeric limits seen in your data)
assert df["Conversion %"].between(1, 100).all()            # a rate can't be below 0 or above 100%
assert (df["Acquisition_Cost"] > 0).all()                # cost must be positive
assert df["ROI %"].between(1, 100).all()                     # stored as a fraction after cleaning
assert (df["Clicks"] >= 0).all()
assert (df["Impressions"] >= 0).all()
assert df["Engagement_Score"].between(1, 10).all()       # 1-10 scale

# 5. Allowed values (categorical columns)
allowed = {
    "Campaign_Type": {"Display", "Email", "Influencer", "Search", "Social Media"},
    "Target_Audience": {"All Ages", "Men 18-24", "Men 25-34", "Women 25-34", "Women 35-44"},
    "Channel_Used": {"Email", "Facebook", "Google Ads", "Instagram", "Website", "YouTube"},
    "Location": {"Chicago", "Houston", "Los Angeles", "Miami", "New York"},
    "Customer_Segment": {"Fashionistas", "Foodies", "Health & Wellness",
                         "Outdoor Adventurers", "Tech Enthusiasts"},
}
for col, valid in allowed.items():
    bad = set(df[col].unique()) - valid
    assert not bad, f"{col} has unexpected values: {bad}"

# 6. Date checks
assert df["Date"].between("2021-01-01", "2021-12-31").all()   # within the expected year
                    # optional: dates ordered by ID

# 7. Text hygiene: no stray spaces or inconsistent casing
for col in allowed:
    assert (df[col] == df[col].str.strip()).all()

# 8. Outliers (flag, don't drop) using the IQR rule
for col in ["Acquisition_Cost", "Clicks","Impressions", "Conversion %", "ROI %"]:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col}: {n_out} outliers")

# 9. Cross-check against the raw file (cleaning didn't lose or change data)
raw = pd.read_csv("marketing_campaign_dataset - raw.csv")
assert len(raw) == len(df)
assert (raw["Campaign_ID"].values == df["Campaign_ID"].values).all()

Acquisition_Cost: 0 outliers
Clicks: 0 outliers
Impressions: 0 outliers
Conversion %: 0 outliers
ROI %: 0 outliers


In [23]:
print("Conversion % mean:", df["Conversion %"].mean().round(2))
print("Acquisition_Cost mean:", df["Acquisition_Cost"].mean().round(0))
print("ROI % mean:", df["ROI %"].mean().round(2))
print("Clicks mean:", df["Clicks"].mean().round(2))
print("Impressions mean:", df["Impressions"].mean().round(2))
print("Engagement_Score mean:", df["Engagement_Score"].mean().round(2))

Conversion % mean: 8.01
Acquisition_Cost mean: 12504.0
ROI % mean: 5.0
Clicks mean: 549.77
Impressions mean: 5507.3
Engagement_Score mean: 5.49


In [30]:
CTR = ((df["Clicks"].sum())/(df["Impressions"].sum())*100)
print(CTR.round(2))


9.98


In [44]:
df["Campaign_ID"].unique()

array([     1,      2,      3, ..., 199998, 199999, 200000],
      shape=(200000,))

In [45]:
df['Campaign_ID'].nunique()

200000

In [48]:
print("Campaign_Type:",df["Campaign_Type"].unique())
print("Channel_Used:",df["Channel_Used"].unique())
print("Target_Audience:",df["Target_Audience"].unique())
print("Location:",df["Location"].unique())
print("Customer_Segment:",df["Customer_Segment"].unique())

Campaign_Type: ['Email' 'Influencer' 'Display' 'Search' 'Social Media']
Channel_Used: ['Google Ads' 'YouTube' 'Instagram' 'Website' 'Facebook' 'Email']
Target_Audience: ['Men 18-24' 'Women 35-44' 'Men 25-34' 'All Ages' 'Women 25-34']
Location: ['Chicago' 'New York' 'Los Angeles' 'Miami' 'Houston']
Customer_Segment: ['Health & Wellness' 'Fashionistas' 'Outdoor Adventurers' 'Foodies'
 'Tech Enthusiasts']
